This example fetches all pages for each username and UTC date. A search can span up to 30 days after its start date. Install `tqdm` and `pandas` in the notebook environment to display progress and build a table.

In [1]:
from datetime import datetime, timedelta


def generate_date_list(start_date: str, end_date: str) -> list[str]:
    """
    Generate a list of dates between two dates, inclusive.

    Args:
        start_date (str): The start date as a string in "YYYYMMDD" format.
        end_date (str): The end date as a string in "YYYYMMDD" format.

    Returns:
        list of str: A list containing all dates from the start date to the end date,
                     inclusive, each formatted as "YYYYMMDD".

    Raises:
        ValueError: If the input date strings are not in the correct format or if
                    the start date occurs after the end date.

    Example:
        >>> generate_date_list("20240811", "20240815")
        ['20240811', '20240812', '20240813', '20240814', '20240815']
    """
    date_format = "%Y%m%d"
    first = datetime.strptime(start_date, date_format)
    last = datetime.strptime(end_date, date_format)
    if last < first:
        raise ValueError("end_date must be on or after start_date")
    delta = last - first
    return [
        (first + timedelta(days=i)).strftime(date_format)
        for i in range(delta.days + 1)
    ]


start = "20240816"
end = "20241003"

dates = generate_date_list(start, end)


We recommend you store your client key and secret in the environment variables and call them here.

In [ ]:
from os import environ
from TikTok.Query import Query
from TikTok.Auth import OAuth2
from TikTok.ValidationModels.OAuth2 import RequestHeadersModel, TokenRequestBodyModel

auth: OAuth2 = await OAuth2.authenticate(
    headers=RequestHeadersModel(),
    body=TokenRequestBodyModel(
        client_key=environ["CLIENT_KEY"],
        client_secret=environ["CLIENT_SECRET"],
    ),
)

query = Query(auth)


The SDK reports HTTP and API errors with `QueryException`. The example retries connection failures, HTTP 429, and server errors. It reports a failed page when the retry limit is reached.

In [ ]:
from TikTok.Exceptions.Query import QueryException
from TikTok.ValidationModels.Video import (
    VideoQueryRequestBuilder,
    VideoQueryOperation,
    VideoQueryFieldName,
    VideoQueryFields,
    VideoDataModel,
)
import asyncio
from tqdm import tqdm


async def fetch_videos(
    username: str, date_list: list[str], max_retries: int = 3
) -> list[VideoDataModel]:
    """
    Fetches videos for a given username over multiple dates, with a progress bar and retry logic.

    Args:
        username (str): The TikTok username to query.
        date_list (list): A list of date strings in "YYYYMMDD" format where each date
                          serves as both the start and end date for the query.
        max_retries (int): Maximum number of retries for failed API calls.

    Returns:
        list: A list containing all fetched VideoDataModel instances.
    """
    all_videos: list[VideoDataModel] = []

    for date in tqdm(date_list, desc="Processing Dates", unit="date"):
        video_query = VideoQueryRequestBuilder()

        current_request = (
            video_query.start_date(date)
            .end_date(date)
            .max_count(100)
            .and_(VideoQueryOperation.EQ, VideoQueryFieldName.username, [username])
            .build()
        )

        retries = 0

        while True:
            try:
                video_query_response = await query.video.search(
                    request=current_request,
                    fields=[
                        VideoQueryFields.id,
                        VideoQueryFields.video_description,
                        VideoQueryFields.create_time,
                        VideoQueryFields.region_code,
                        VideoQueryFields.share_count,
                        VideoQueryFields.view_count,
                        VideoQueryFields.like_count,
                        VideoQueryFields.comment_count,
                        VideoQueryFields.music_id,
                        VideoQueryFields.hashtag_names,
                        VideoQueryFields.username,
                        VideoQueryFields.effect_ids,
                        VideoQueryFields.playlist_id,
                        VideoQueryFields.favorites_count,
                    ],
                )
            except QueryException as error:
                status = error.status_code
                if retries >= max_retries or (
                    status is not None and status != 429 and status < 500
                ):
                    raise
                retries += 1
                await asyncio.sleep(2**retries)
                continue

            retries = 0
            page = video_query_response.data
            all_videos.extend(page.videos)
            if not page.has_more:
                break
            if page.cursor is None or not page.search_id:
                raise ValueError("TikTok omitted the cursor or ID for the next page.")
            current_request = current_request.model_copy(
                update={"cursor": page.cursor, "search_id": page.search_id}
            )

    return all_videos


If you want to fetch videos from multiple users, you can do it as follows:

In [ ]:
async def fetch_videos_for_usernames(
    usernames: list[str], date_list: list[str], max_retries: int = 3
) -> list[VideoDataModel]:
    """
    Fetches videos for multiple usernames over multiple dates, with nested progress bars.

    Args:
        usernames (list[str]): A list of TikTok usernames to query.
        date_list (list[str]): A list of date strings in "YYYYMMDD" format where each date
                          serves as both the start and end date for the query.
        max_retries (int): Maximum number of retries for failed API calls.

    Returns:
        list: A list containing all fetched VideoDataModel instances from all usernames.
    """
    all_videos: list[VideoDataModel] = []

    for username in tqdm(usernames, desc="Processing Usernames", unit="username"):
        print(f"Processing: {username}")
        user_videos = await fetch_videos(username, date_list, max_retries)
        all_videos.extend(user_videos)

    return all_videos


In [ ]:
profiles = ["tiktok"]
videos = await fetch_videos_for_usernames(profiles, dates)


Convert the selected video fields to a table. Missing values represent fields that TikTok did not return.

In [ ]:
import pandas as pd

df = pd.DataFrame.from_records(
    [video.model_dump(by_alias=True, exclude_none=True) for video in videos]
)

await query.client.aclose()
await auth.client.aclose()
